In [1]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
import os

In [2]:
# 使用小米 MiMo 模型
llm = ChatOpenAI(
    model="mimo-v2.5-pro",
    base_url="https://token-plan-cn.xiaomimimo.com/v1",
    api_key=os.environ.get("MIMO_API_KEY"),
    temperature=0.7
)

In [3]:
# --- 定义反思链的三个组件 ---

# 1. 生成链：创建初稿
generation_chain = (
    ChatPromptTemplate.from_messages([
        ("system", "为智能咖啡杯写一个简短的产品描述"),
        ("user", "{product_details}")
    ])
    | llm
    | StrOutputParser()
)

# 2. 批评链：评估描述
critique_chain = (
    ChatPromptTemplate.from_messages([
        ("system", """批评这个产品描述，提供改进建议：
        - 清晰度
        - 简洁性
        - 吸引力"""),
        ("user", "产品描述：\n{initial_description}")
    ])
    | llm
    | StrOutputParser()
)

# 3. 改进链：根据批评重写
refinement_chain = (
    ChatPromptTemplate.from_messages([
        ("system", "根据批评重写产品描述，保持优点，改进缺点"),
        ("user", "原始描述：\n{initial_description}\n\n批评：\n{critique}")
    ])
    | llm
    | StrOutputParser()
)


In [4]:
# --- 执行反思流程 ---

# 输入产品信息
product_details = "智能咖啡杯，具有保温功能，可通过APP控制温度"

print("--- 步骤1: 生成初始描述 ---")
initial_description = generation_chain.invoke({"product_details": product_details})
print(initial_description)

print("\n--- 步骤2: 批评评估 ---")
critique = critique_chain.invoke({"initial_description": initial_description})
print(critique)

print("\n--- 步骤3: 改进描述 ---")
refined_description = refinement_chain.invoke({
    "initial_description": initial_description,
    "critique": critique
})
print(refined_description)


--- 步骤1: 生成初始描述 ---
# ☕ 智能恒温咖啡杯

## 每一口，都是刚刚好的温度

告别"咖啡凉了"的烦恼！这款智能咖啡杯，让每一口饮品都保持在您最喜爱的温度。

---

### 🔥 核心亮点

- **精准控温**：支持 **40°C–65°C** 自由调节，通过手机APP一键设定，找到属于您的完美口感温度。
- **持久保温**：内置高效保温芯片，长效恒温可达 **8小时**，从清晨到午后，温暖始终如一。
- **APP智能操控**：连接专属APP，实时显示杯内温度，远程调节加热档位，还可设置个性化饮温提醒。
- **LED温度显示**：杯身配备隐藏式温度显示屏，轻触即显，温度一目了然。
- **食品级材质**：采用304不锈钢内胆 + 防烫外壳，安全环保，易清洗。

---

### 🎯 适用场景

> 办公桌上的专注时刻 · 深夜书房的灵感时光 · 差旅途中的温暖陪伴

---

### 📦 产品参数

| 项目 | 详情 |
|------|------|
| 容量 | 380ml |
| 续航 | 约8小时（恒温模式） |
| 充电 | Type-C 磁吸充电 |
| 重量 | 约320g |
| 兼容系统 | iOS / Android |

---

**智能生活，从一杯好咖啡开始。** ☕✨

--- 步骤2: 批评评估 ---
您提供的产品描述结构清晰、信息完整，但在**清晰度、简洁性和吸引力**方面仍有提升空间。以下是具体的批评与改进建议：

---

### **一、清晰度**
**问题：**
1.  **核心亮点表述偏技术化**：如“高效保温芯片”“隐藏式温度显示屏”等，对普通用户而言不够直观。
2.  **温度范围说明不足**：40°C–65°C 是什么概念？用户可能不清楚不同温度对应什么饮品（如40°C适合温水，65°C适合热咖啡）。
3.  **适用场景较笼统**：仅用短语描述，缺乏具体情境或用户痛点的连接。

**改进建议：**
- **用生活化语言解释技术参数**：  
  例如：“精准控温（40°C–65°C）——无论是45°C的温水、55°C的拿铁，还是65°C的热美式，都能一键设定，匹配你最爱的口感。”
- **补充温度与饮品的对应关系**：  
  在控温说明后增加小贴士：“40-50°C：温水/花茶 |

In [5]:
# --- 扩展：循环反思 ---
# 多次执行反思流程，持续改进

product_details = "智能咖啡杯，具有保温功能，可通过APP控制温度"

# 定义循环反思函数
def iterative_reflection(product_details, iterations=3):
    """执行多次反思迭代"""
    current_description = None
    
    for i in range(iterations):
        print(f"\n{'='*50}")
        print(f"--- 第 {i+1} 轮反思 ---")
        print(f"{'='*50}")
        
        # 第一轮使用产品详情，后续使用上一轮的改进结果
        if i == 0:
            print("\n--- 生成初始描述 ---")
            current_description = generation_chain.invoke({"product_details": product_details})
        else:
            print(f"\n--- 基于第 {i} 轮批评进行改进 ---")
            current_description = refinement_chain.invoke({
                "initial_description": current_description,
                "critique": critique
            })
        
        print(current_description)
        
        # 批评评估
        print(f"\n--- 第 {i+1} 轮批评 ---")
        critique = critique_chain.invoke({"initial_description": current_description})
        print(critique)
    
    # 最终改进
    print(f"\n{'='*50}")
    print("--- 最终改进 ---")
    print(f"{'='*50}")
    final_description = refinement_chain.invoke({
        "initial_description": current_description,
        "critique": critique
    })
    
    return final_description

# 执行3轮反思
final_result = iterative_reflection(product_details, iterations=3)

print(f"\n{'='*50}")
print("--- 最终结果 ---")
print(f"{'='*50}")
print(final_result)



--- 第 1 轮反思 ---

--- 生成初始描述 ---
# ☕ 智能温控咖啡杯 — 每一口，都是刚好的温度

## 产品简介

告别反复加热的烦恼，让每一口咖啡都恰到好处。

这款智能咖啡杯采用**双层真空保温技术**，搭配精准温控芯片，可将饮品温度稳定保持在 **40°C–65°C** 之间，长达 **3小时**。无论是清晨的拿铁，还是午后的红茶，始终为您锁定最佳饮用温度。

## 核心亮点

- 📱 **APP智能控温**：通过专属手机APP，一键设定您偏好的温度，实时监控杯内温度变化
- 🌡️ **精准恒温**：内置NTC温度传感器，温控精度达±1°C
- 🔋 **无线充电底座**：磁吸式充电设计，放置即充电，满电续航约8小时
- ⏰ **定时提醒**：可设置饮水提醒，养成健康饮水习惯
- 🛡️ **安全材质**：316不锈钢内胆，食品级安全，BPA-free

## 产品参数

| 项目 | 参数 |
|------|------|
| 容量 | 380ml |
| 材质 | 316不锈钢 + 食品级硅胶 |
| 温控范围 | 40°C – 65°C |
| 续航时间 | 约8小时 |
| 充电方式 | 磁吸无线充电 |
| 适配系统 | iOS / Android |

## 适用场景

> 办公室 ☕ 居家 🏠 阅读 📖 写作 ✍️

---

**一杯好咖啡，值得被认真对待。**
智能温控，让温度懂你心意。 🫶

--- 第 1 轮批评 ---
### 批评与改进建议

#### 1. **清晰度**
- **优点**：结构清晰，分模块展示产品信息，参数表格一目了然。
- **可改进处**：
  - 技术术语（如“NTC温度传感器”“双层真空保温”）对普通用户可能不够直观，可简要说明其作用（例如：“精准感知温度变化”“高效锁温”）。
  - 温控范围（40°C–65°C）可补充说明适合的饮品类型（如：40°C适合热巧克力，60°C适合拿铁），帮助用户理解适用场景。

#### 2. **简洁性**
- **优点**：文案整体简洁，无冗余信息。
- **可改进处**：
  - “产品简介”部分稍显技术化，可更侧重用户体验。例如将“采用双层真空保温技术……”简化为“长效锁温3小时，无需反复加热”。
  - 核心亮点中“APP智能控温”和“精准恒温